# Limpieza de Departamentos

Lee las **primeras 3 hojas** del Excel.  
- Nombre de hoja → **Company**  
- `SITIO` → **Site**  
- `TIENDA` → **Depart**  
- `ESTACION` → se descarta

In [22]:
import pandas as pd
import re

EXCEL_FILE = r"C:\Users\RTX\Documents\GitHub\PT\PROYECTO_TERMINAL_INVENTARIO_TI\datos\DEPARTAMENTOS_2.xlsx"   # <-- cambia por tu ruta real

## 1 · Correcciones de texto conocidas

Agrega más entradas al diccionario si encuentras caracteres raros o typos.

In [23]:
FIXES = {
    "PIER27 CARIBE„O": "PIER27 CARIBEÑO",
    # "TEXTO MAL":  "TEXTO CORRECTO",
}

## 2 · Función de limpieza de texto

In [24]:
def clean_text(value):
    """strip · colapsa espacios · UPPER · aplica FIXES"""
    if not isinstance(value, str):
        return value
    v = value.strip()
    v = re.sub(r"\s+", " ", v)
    v = v.upper()
    return FIXES.get(v, v)

## 3 · Función de limpieza por hoja

In [25]:
def clean_sheet(df, company_name):
    df = df.drop(columns=["ESTACION"], errors="ignore").copy()
    df = df.dropna(subset=["SITIO", "TIENDA"])
    df["SITIO"]  = df["SITIO"].apply(clean_text)
    df["TIENDA"] = df["TIENDA"].apply(clean_text)
    df = df.drop_duplicates(subset=["SITIO", "TIENDA"])
    df.insert(0, "COMPANY", company_name)
    return df.reset_index(drop=True)

## 4 · Leer y limpiar las primeras 3 hojas

In [26]:
xf            = pd.ExcelFile(EXCEL_FILE)
first_3_names = xf.sheet_names[:3]        # solo las primeras 3 hojas
raw           = pd.read_excel(EXCEL_FILE, sheet_name=first_3_names)




In [28]:
df_clean = pd.concat(
    [clean_sheet(raw[name], name) for name in first_3_names],
    ignore_index=True
)

df_clean

,COMPANY,SITIO,TIENDA
0,ISLA17(42),BEACHPALACE,TABAQUERIA
1,ISLA17(42),BEACHPALACE,JOYERIA
2,ISLA17(42),*CUN T2 PA,CORONA
3,ISLA17(42),*CUN T2 PB,CORONA
4,ISLA17(42),CUN T3 ISLA C11,CORONA
...,...,...,...
88,PTOARENAS,SIRENIS,CARRETA
89,PTOARENAS,PALMAR BEACH,PIER27
90,PTOARENAS,PALMAR BEACH,AMIANI
91,PTOARENAS,SIRENIS,VIVA MEXICO


## 5 · Resumen por company

In [29]:
resumen = df_clean.groupby("COMPANY")[["SITIO", "TIENDA"]].nunique()
resumen.columns = ["sites", "departs"]
resumen["total_rows"] = df_clean.groupby("COMPANY").size()
resumen

,sites,departs,total_rows
COMPANY,,,
HSPRO(33),22,6,35
ISLA17(42),13,13,29
PTOARENAS,12,10,29


## 6 · Exportar (opcional)

In [30]:
df_clean.to_csv("departamentos_limpio.csv", index=False, encoding="utf-8-sig")
print(f"Exportado: {len(df_clean)} filas")

Exportado: 93 filas
